In [12]:
!pip install sentence-transformers faiss-cpu requests


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import requests

In [36]:
with open("corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus size:", len(corpus))

Corpus size: 20882


In [37]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3487.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
corpus_embeddings = model.encode(corpus, show_progress_bar=True)

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 653/653 [01:30<00:00,  7.22it/s]


In [39]:
dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(corpus_embeddings))

print("FAISS index built:", index.ntotal)

FAISS index built: 20882


In [40]:
faiss.write_index(index, "faiss_index.index")

In [41]:
def web_search(query, max_results=5):

    keywords = " ".join(query.split()[:6])
    query = keywords + " news"

    url = "https://api.duckduckgo.com/"

    params = {
        "q": query,
        "format": "json",
        "no_html": 1
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
    except:
        return []

    results = []

    if data.get("AbstractText"):
        results.append(data["AbstractText"])

    for topic in data.get("RelatedTopics", []):
        if isinstance(topic, dict) and topic.get("Text"):
            results.append(topic["Text"])

    return results[:max_results]

In [42]:
def web_search_mock(query):
    return [
        "A recent industrial accident disrupted supply chains globally.",
        "Reports indicate a refinery explosion affected fuel availability.",
        "Government advisory issued to prevent panic buying."
    ]

In [43]:
def clean_web_results(results):

    cleaned = []

    for r in results:
        if len(r.split()) < 5:
            continue
        cleaned.append(r.strip())

    return list(set(cleaned))

In [44]:
import json

def add_to_corpus(new_texts):

    global corpus, index

    # 🔥 remove duplicates
    new_texts = [t for t in new_texts if t not in corpus]

    if not new_texts:
        print("⚠️ No unique data to add")
        return

    before = len(corpus)

    corpus.extend(new_texts)

    # 🔥 UPDATE FAISS
    new_embeddings = model.encode(new_texts)
    index.add(np.array(new_embeddings))

    after = len(corpus)

    print(f"Added {after - before} new entries")
    print(f"Corpus size: {before} → {after}")
    print(f"FAISS size: {index.ntotal}")

    # 🔥 SAVE TO FILE (IMPORTANT)
    with open("corpus.json", "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)

    print("💾 Corpus saved to file")

In [45]:
def hybrid_retrieve(claim, top_k=5, threshold=0.65, use_mock=False):

    claim_embedding = model.encode([claim])

    distances, indices = index.search(np.array(claim_embedding), top_k)

    similarity = 1 / (1 + distances[0][0])

    local_results = [corpus[i] for i in indices[0]]

    print("\n==============================")
    print("CLAIM:", claim)
    print("Similarity:", round(similarity, 3))

    # ✅ GOOD MATCH
    if similarity >= threshold:
        print("Using FAISS corpus")
        return local_results

    # 🌐 FALLBACK
    print("Using WEB fallback")
    
    web_results = web_search_serpapi(claim)

    web_results = clean_web_results(web_results)

    print("Web results:", len(web_results))

    add_to_corpus(web_results)

    return web_results if web_results else local_results

In [46]:
claims = [
    # 🔥 Synthetic + very specific (guaranteed not in corpus)
    "A hydrogen-powered cargo ship exploded in the Arctic Ocean in February 2026 disrupting international trade routes",
    "A private space station experienced a critical oxygen failure in orbit in 2026 forcing emergency evacuation",
    "A global AI blackout occurred in 2026 due to a failure in distributed neural network infrastructure",

    # 🔥 Fiction-like but realistic (excellent for testing)
    "A quantum internet experiment in Switzerland caused temporary global encryption instability in 2026",
    "An underground data center fire in Iceland disrupted cloud services across Europe in 2025",
    "A solar flare in 2026 damaged satellite communication networks leading to global internet outages",

    # 🔥 Complex multi-entity claims
    "Japan deployed autonomous AI-controlled traffic systems nationwide in 2026 reducing urban congestion by 40 percent",
    "A biotech firm in Germany successfully created lab-grown human organs for transplant in 2026",
    "A global financial crash in 2025 was triggered by AI-driven stock trading algorithms malfunctioning",

    # 🔥 Policy + futuristic mix
    "The European Union banned fully autonomous weapons systems in 2026 citing ethical concerns",
    "India launched a nationwide AI-powered surveillance grid in 2026 integrating facial recognition across all cities",

    # 🔥 Scientific edge cases
    "Scientists achieved room-temperature superconductivity at atmospheric pressure in 2026",
    "A fusion reactor in South Korea generated sustained net-positive energy for 48 hours in 2026",

    # 🔥 Rare combinations
    "A deep-sea mining accident in the Pacific Ocean in 2026 caused significant ecological damage",
    "A global pandemic simulation AI system predicted a new outbreak scenario with 92 percent accuracy in 2025"

]

In [47]:
for claim in claims:
    results = hybrid_retrieve(claim, use_mock=False)

    print("\nEvidence:")
    for r in results:
        print("→", r)


CLAIM: A hydrogen-powered cargo ship exploded in the Arctic Ocean in February 2026 disrupting international trade routes
Similarity: 0.522
Using WEB fallback
Web results: 5
⚠️ No unique data to add

Evidence:
→ The Arctic Metagaz incident should not be treated as an isolated maritime incident. It warns us of the growing risks embedded in the global ...
→ The impact has been brutal: oil prices have surged over 50%, hammering economies everywhere with skyrocketing fuel costs, supply chain chaos, ...
→ On March 3, 2026, the Russian-flagged LNG tanker Arctic Metagaz exploded and subsequently sank in the Mediterranean Sea. The vessel was located ...
→ A fully laden Russian flagged 'shadow fleet' tanker has been hit by uncrewed surface vessels (USVs - maritime drones) just East of Malta.
→ The Russian flag tanker Arctic Meta Gas which was transporting liqufied natural gas has caught fire in the Mediterranean Sea near Malta in Lebanon.

CLAIM: A private space station experienced a critical o

In [48]:
print("\nFinal corpus size:", len(corpus))
print("FAISS size:", index.ntotal)

print("\nLast entries:")
for c in corpus[-5:]:
    print("→", c)


Final corpus size: 20887
FAISS size: 20887

Last entries:
→ These environments demand reliability, cybersecurity, and intelligent monitoring at scale, reinforcing that AI-powered CCTV is now foundational ...
→ Delhi Police to Deploy AI-Enabled Smart Glasses with Facial Recognition for Republic Day Security JAN 21, 2026 (AGENCIES): The Delhi Police ...
→ Sparsh CCTV is India's leading indigenous video surveillance manufacturer, offering a comprehensive portfolio of AI-powered, cyber-secured, and ...
→ In Lucknow, Uttar Pradesh, AI-enabled cameras under the Safe City Project generate real-time alerts to law enforcement when they detect “subtle ...
→ Over 500 AI- enabled CCTV cameras equipped with facial recognition and ANPR systems are operational, monitored from a dedicated control room.


In [25]:
!pip install google-search-results

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for google-search-results: started
  Running setup.py install for google-search-results: finished with status 'done'


  DEPRECATION: google-search-results is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from serpapi import GoogleSearch

def web_search_serpapi(query, max_results=5):

    params = {
        "q": query,
        "api_key": "a1998f816e6a8c676e95c504b8b0620ecfde0eab89b4fff491169872753f4093",
        "num": max_results
    }

    search = GoogleSearch(params)
    results = search.get_dict()

    snippets = []

    for r in results.get("organic_results", []):
        if "snippet" in r:
            snippets.append(r["snippet"])

    return snippets[:max_results]

In [34]:
import json

with open("corpus.json", "r", encoding="utf-8") as f:
    corpus_file = json.load(f)

print("Corpus size from file:", len(corpus_file))

Corpus size from file: 20882
